In [ ]:
import os
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from dotenv import load_dotenv

load_dotenv()

from huggingface_hub import login
API_KEY = os.getenv("HF_KEY") # some models require a huggingface access token
if not API_KEY:
    raise ValueError("API key not found. Did you create .env containing a key?")
login(API_KEY)

In [ ]:
print("’".encode("unicode_escape"))
print("”".encode("unicode_escape"))
print("“".encode("unicode_escape"))
# quotation mark characters to be replaced with ' and "

b'\\u2019'
b'\\u201d'
b'\\u201c'


In [ ]:
# read lines of knowledge base, replace characters, and strip whitespace
knowledge_base = []
with open('./knowledge.txt') as knowledge_in:
    for line in knowledge_in:
        knowledge_base.append(line.replace('\u201d','"').replace('\u201c','"').replace('\u2019',"'").rstrip())
# lines stored in list

In [ ]:
embedder = SentenceTransformer("google/embeddinggemma-300m")

# embeddings are stored in vector database for fast similarity search
client = chromadb.Client()
collection = client.get_or_create_collection("rag_knowledge_base")
collection.add(
    documents=knowledge_base,
    ids=[str(uuid.uuid4()) for _ in knowledge_base],
    embeddings=embedder.encode(knowledge_base)
)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 9635.02it/s]


In [ ]:
# get k most relevant knowledge base entries to the query
def retrieve_context(query, top_k=3):
    # embed query
    query_embedding = embedder.encode([query]).tolist()
    # query database with that embedding
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0]

In [ ]:
# prompt to be fed to the response generator
# context is the knowledge base entries from retrieve_context
def build_prompt(context, question):
    return f'''
Use the following context to answer the question.

Context:
{context}

Question:
{question}
'''

In [ ]:
# for response generation
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 12797.30it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 80) -> str:
    """Generate an answer from a seq2seq model (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # deterministic
            num_beams=1
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

In [ ]:
query = "What are the sizes of motherboard?"
context = retrieve_context(query, top_k=5) # top 5 entries seems to work better than 3
generate_answer(build_prompt(context, query))

'ATX (full-size, mainstream motherboard), MicroATX (smaller with fewer expansion slots and other surface-area-related features), and Mini-ITX, (even smaller, usually just a single slot for a video card), XL-ATX and EATX, which are bigger than ATX'